In [ ]:
import feedparser
from openai import OpenAI
import datetime
from dateutil import parser as dateparser
import json
import requests
import os
from bs4 import BeautifulSoup # For cleaning HTML from summaries
from dotenv import load_dotenv

# ------------- CONFIG ----------------

# Load environment variables from .env file (must be in the same directory)
load_dotenv()
   
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
  
if not OPENAI_API_KEY:
	raise ValueError("OPENAI_API_KEY not found. Make sure it's set in your .env file.")

# Use the modern OpenAI client
client = OpenAI(api_key=OPENAI_API_KEY)

# Using gpt-4o-mini: faster, more capable, and cost-effective for this task.
LLM_MODEL = "gpt-4o-mini"

# --- User-Curated RSS Feeds ---
RSS_FEEDS = {

    # NEP - Economics
    "NEP-FOR": "https://nep.repec.org/rss/nep-for.rss.xml",   # **Forecasting** (Most Important)
    "NEP-ETS": "https://nep.repec.org/rss/nep-ets.rss.xml",   # Econometric Time Series
    "NEP-BIG": "https://nep.repec.org/rss/nep-big.rss.xml",   # Big Data
    "NEP-CMP": "https://nep.repec.org/rss/nep-cmp.rss.xml",   # Computational Economics
    "NEP-MAC": "https://nep.repec.org/rss/nep-mac.rss.xml",   # Macroeconomics
    "NEP-MON": "https://nep.repec.org/rss/nep-mon.rss.xml",   # Monetary Economics
    "NEP-MFD": "https://nep.repec.org/rss/nep-mfd.rss.xml",   # Macrofinance
    "NEP-CBA": "https://nep.repec.org/rss/nep-cba.rss.xml",   # Central Banking
    "NEP-AIN": "https://nep.repec.org/rss/nep-ain.rss.xml",   # Artificial Intelligence
    "NEP-IFN": "https://nep.repec.org/rss/nep-ifn.rss.xml",   # International Finance
    
    # NEP - Finance
    "NEP-FMK": "https://nep.repec.org/rss/nep-fmk.rss.xml",   # Financial Markets
    "NEP-RMG": "https://nep.repec.org/rss/nep-rmg.rss.xml",   # Risk Management
    "NEP-PPM": "https://nep.repec.org/rss/nep-ppm.rss.xml",   # Project, Program and Portfolio Management

    # NEP - Energy & Resource Focused
    "NEP-ENE": "https://nep.repec.org/rss/nep-ene.rss.xml",   # Energy Economics
    "NEP-RES": "https://nep.repec.org/rss/nep-res.rss.xml",   # Resource Economics

    # ArXiv - Core Quantitative & CS Fields
    "arXiv-econEM": "http://export.arxiv.org/rss/econ.EM",     # Econometrics
    "arXiv-statML": "http://export.arxiv.org/rss/stat.ML",     # Statistics - Machine Learning
    "arXiv-csLG": "http://export.arxiv.org/rss/cs.LG",         # Computer Science - Machine Learning
    "arXiv-qFIN-ST": "http://export.arxiv.org/rss/q-fin.ST",   # Quantitative Finance - Statistical Methods
    "arXiv-qFIN-CP": "http://export.arxiv.org/rss/q-fin.CP",   # Quantitative Finance - Computational Finance

    # High-Signal Institutions (Central Banks & Academic Orgs)
    "FEDS_Papers": "https://www.federalreserve.gov/feeds/working_papers.xml",     # US Federal Reserve Board
    "ECB_Papers": "https://www.ecb.europa.eu/rss/pub.html",                       # European Central Bank
    "IMF_Papers": "https://www.imf.org/en/Publications/RSS?language=eng&series=IMF%20Working%20Papers", # International Monetary Fund
    "BIS_Papers": "https://www.bis.org/doclist/bis_fsi_publs.rss",                # Bank for International Settlements
    "CEPR_Papers": "https://cepr.org/rss/discussion-paper",                       # Centre for Economic Policy Research (CEPR)
}

SCORE_THRESHOLD = 7.0  # Keep only papers scoring above this
DAYS_BACK = 7          # Look back this many days
OUTPUT_DIR = "newsletters"

# ------------- HELPER FUNCTIONS ----------------

def _get_authors(entry):
    """Normalizes author information from a feed entry."""
    if hasattr(entry, 'authors') and entry.authors:
        return ', '.join(author['name'] for author in entry.authors if 'name' in author)
    if hasattr(entry, 'author'):
        return entry.author
    return "Unknown"

# ------------- CORE FUNCTIONS ----------------

def fetch_recent_papers():
    """Fetch recent papers from RSS feeds with robust date parsing and encoding correction."""
    print(f"Fetching papers from {len(RSS_FEEDS)} sources, looking back {DAYS_BACK} days...")
    cutoff_date = datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(days=DAYS_BACK)
    papers = []

    for source, url in RSS_FEEDS.items():
        print(f"  - Processing {source} from {url}")
        try:
            # ---- FIX: Fetch manually to override bad charset headers ----
            response = requests.get(url, timeout=10)

            # If feed declares UTF-8 in content but server says otherwise, force UTF-8
            if "utf-8" in response.text.lower() or "<?xml" in response.text:
                response.encoding = "utf-8"  # Force correct decoding

            feed = feedparser.parse(response.text)

            if feed.bozo:
                print(f"    Warning: Malformed feed for {source}. Error: {feed.bozo_exception}")

            found_in_feed = 0
            for entry in feed.entries:
                pub_date_str = getattr(entry, "published", getattr(entry, "updated", None))
                if not pub_date_str:
                    continue
                try:
                    pub_date = dateparser.parse(pub_date_str)
                except dateparser.ParserError:
                    continue
                if pub_date.tzinfo is None:
                    pub_date = pub_date.replace(tzinfo=datetime.timezone.utc)
                if pub_date < cutoff_date:
                    continue

                summary = getattr(entry, "summary", "")
                if "<" in summary and ">" in summary:
                    summary = BeautifulSoup(summary, "html.parser").get_text(separator=' ', strip=True)

                papers.append({
                    "title": entry.title,
                    "link": entry.link,
                    "summary": summary,
                    "authors": _get_authors(entry),
                    "source": source,
                    "date": pub_date.strftime("%Y-%m-%d")
                })
                found_in_feed += 1

            print(f"    Found {found_in_feed} recent papers in {source}.")

        except Exception as e:
            print(f"    Error fetching or parsing feed {source}: {e}")
            continue

    return papers

def score_and_summarize(paper):
    """Sends abstract to LLM for summary, scoring, and categorization."""
    prompt = f"""
You are a highly specialized expert curator for "ets4 Weekly (Economic Time Series Forecasting Weekly)," a newsletter focused **exclusively on practical and impactful forecasting of economic time series.** Your role is to identify only the most relevant and innovative work for an audience of researchers and practitioners.

You will be given a research paper. Evaluate it strictly through the lens of **forecasting relevance and potential to advance economic prediction.** Papers that focus on **structural analysis, causal inference, descriptive statistics, or purely theoretical econometrics without a predictive component must be classified as Not Relevant**, even if they appear in economics journals.

---

**Paper Information**

Title: {paper['title']}
Authors: {paper['authors']}
Source: {paper['source']}
Date: {paper['date']}
Abstract: {paper['summary']}

---

### Your Tasks

1. **Summarize (3-4 sentences)**  
   Focus only on elements related to **forecasting or predictive modeling**. If forecasting is not clearly present, explicitly say so.

2. **Assign a Quality Score (1–10)**  
   Score based on **forecasting methodological novelty, empirical rigor, and potential practical impact.**  
   - **Extra weight should be given** to papers that:
     - Introduce **new modeling approaches** or **improve existing ones significantly**.
     - Use **novel datasets** that could meaningfully enhance economic forecasting.
     - Present **forecasting applications with real-world decision value** (e.g. monetary policy, energy prices, financial trading, inflation nowcasting).

3. **Classify into One Category**  
   Choose exactly one:
   - **Directly Relevant** → The paper is clearly about **forecasting *economic* time series** and offers meaningful contributions or applications.
   - **Paper of Interest** → The paper is *not economic*, but presents **highly innovative forecasting methods or data strategies** that could plausibly transfer to economics.
   - **Not Relevant** → No clear predictive component, or relevance is too indirect (e.g., **structural models, policy simulations without forecasting, causal effects estimation, variance decompositions, statistical/econometric theory without prediction focus**).

---

### Scoring & Categorization Rules

| Score | Meaning | Default Category |
|--------|---------|------------------|
| **1–3** | Low quality OR no forecasting at all | Not Relevant |
| **4–6** | Minor/incremental forecasting contribution OR niche application with little general value | Not Relevant |
| **7–8** | Solid contribution to forecasting | Directly Relevant *if economic*, otherwise Paper of Interest |
| **9–10** | Breakthrough forecasting methodology, dataset, or application with clear future impact | Directly Relevant *if economic*, otherwise Paper of Interest |

---

### JSON Output Format

Return your response strictly in **valid JSON** using the following schema:

{{
"summary": "...",
"score": X,
"category": "Directly Relevant" | "Paper of Interest" | "Not Relevant",
"adaptability_reason": null | "Required only if category is 'Paper of Interest'"
}}

- If you select **Paper of Interest**, you **must** fill in `"adaptability_reason"` with a short statement explaining how the method could be applied to economic forecasting.
- For **Directly Relevant** or **Not Relevant**, set `"adaptability_reason": null`.
"""
    try:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2,
            response_format={"type": "json_object"}
        )
        content = response.choices[0].message.content
        data = json.loads(content)
        
        # --- Validation of the more complex JSON structure ---
        required_keys = ["summary", "score", "category", "adaptability_reason"]
        if not all(key in data for key in required_keys):
            raise ValueError(f"LLM response missing one or more required keys: {required_keys}")
        
        if not isinstance(data["score"], (int, float)):
            raise ValueError("LLM score is not a number.")
        
        if data["category"] == "Paper of Interest" and not data.get("adaptability_reason"):
            raise ValueError("Category is 'Paper of Interest' but adaptability_reason is missing.")

        data["score"] = float(data["score"])
        return data
        
    except Exception as e:
        print(f"Error during LLM call for paper '{paper['title']}': {e}")
        # Return a default error structure that won't pass the filter
        return {
            "summary": f"Error during processing: {e}", "score": 0.0,
            "category": "Not Relevant", "adaptability_reason": None
        }

def build_markdown(relevant_papers, interest_papers, filename):
    """Creates a markdown file with two distinct sections for shortlisted papers."""
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    today_str = datetime.date.today().strftime('%Y-%m-%d')
    lines = [
        f"# ets4 Weekly – {today_str}\n",
        "A curated list of recent papers on novel methods for forecasting economic time series.\n"
    ]

    # --- Section 1: Directly Relevant Papers ---
    lines.append("## Directly Relevant Papers\n")
    if not relevant_papers:
        lines.append("No directly relevant papers met the criteria this week.\n")
    else:
        relevant_papers.sort(key=lambda p: p['score'], reverse=True)
        for p in relevant_papers:
            lines.append(f"### [{p['title']}]({p['link']})")
            lines.append(f"- **Authors:** {p['authors']}")
            lines.append(f"- **Source:** {p['source']} ({p['date']})")
            lines.append(f"- **Quality Score:** {p['score']:.1f}/10")
            lines.append(f"- **Summary:** {p['summary']}\n")

    # --- Section 2: Papers of Interest ---
    lines.append("\n---\n") # Add a separator
    lines.append("## Papers of Interest\n")
    lines.append("_Methodologically novel papers from other fields that could be adapted for economic forecasting._\n")
    if not interest_papers:
        lines.append("No papers of interest were identified this week.\n")
    else:
        interest_papers.sort(key=lambda p: p['score'], reverse=True)
        for p in interest_papers:
            lines.append(f"### [{p['title']}]({p['link']})")
            lines.append(f"- **Authors:** {p['authors']}")
            lines.append(f"- **Source:** {p['source']} ({p['date']})")
            lines.append(f"- **Quality Score:** {p['score']:.1f}/10")
            lines.append(f"- **Summary:** {p['summary']}")
            lines.append(f"- **Reason for Interest:** {p.get('adaptability_reason', 'N/A')}\n")

    with open(filename, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"✅ Markdown file saved: {filename}")

# ------------- MAIN ----------------

if __name__ == "__main__":
    print("Starting Economic Forecasting Newsletter Pipeline...")
    
    all_papers = fetch_recent_papers()
    print(f"Found {len(all_papers)} total recent papers.")

    unique_papers = []
    seen_links = set()
    for paper in all_papers:
        if paper['link'] not in seen_links:
            unique_papers.append(paper)
            seen_links.add(paper['link'])
    
    if len(all_papers) > len(unique_papers):
        print(f"Removed {len(all_papers) - len(unique_papers)} duplicates. Processing {len(unique_papers)} unique papers.")

    # --- Create two lists for the two categories ---
    shortlisted_relevant = []
    shortlisted_interest = []

    for i, paper in enumerate(unique_papers, 1):
        print(f"Processing paper {i}/{len(unique_papers)}: '{paper['title'][:70]}...'")
        result = score_and_summarize(paper)
        paper.update(result) # Add all new fields (summary, score, category, etc.) to the paper dict

        # --- Route the paper to the correct list based on score and category ---
        if paper.get("score", 0.0) >= SCORE_THRESHOLD:
            if paper.get("category") == "Directly Relevant":
                shortlisted_relevant.append(paper)
                print(f"  -> Shortlisted! (Relevant) Score: {paper['score']:.1f}/10")
            elif paper.get("category") == "Paper of Interest":
                shortlisted_interest.append(paper)
                print(f"  -> Shortlisted! (Interest) Score: {paper['score']:.1f}/10")
            else:
                print(f"  -> Skipped. Category: {paper.get('category', 'Unknown')}, Score: {paper['score']:.1f}/10")
        else:
            print(f"  -> Skipped. Score below threshold: {paper.get('score', 0.0):.1f}/10")

    today_str = datetime.date.today().strftime("%Y-%m-%d")
    output_filename = os.path.join(OUTPUT_DIR, f"ets4_weekly_{today_str}.md")
    
    # --- Pass both lists to the updated markdown builder ---
    build_markdown(shortlisted_relevant, shortlisted_interest, output_filename)
    
    print("\n--- Pipeline Finished ---")
    print(f"Total unique papers processed by LLM: {len(unique_papers)}")
    print(f"Shortlisted {len(shortlisted_relevant)} 'Directly Relevant' papers.")
    print(f"Shortlisted {len(shortlisted_interest)} 'Papers of Interest'.")